# PicoCal — Training-ready dataset (mentor feedback 25/06)

## Setup

In [1]:
from pathlib import Path
import collections
import numpy as np
import awkward as ak
import uproot
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
root_files = sorted((repo / "data" / "full").glob("matched_*.root"))
INPUT_FILE = root_files[0]
TREE_NAME = "clusters_matched"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

WINDOW = 3
USE_TIMING = False
POS_REF = "seed"
VERTEX_Z_MAX = 100.0

{"input": INPUT_FILE.name, "n_files": len(root_files), "device": str(DEVICE),
 "WINDOW": WINDOW, "USE_TIMING": USE_TIMING, "POS_REF": POS_REF, "VERTEX_Z_MAX": VERTEX_Z_MAX}

{'input': 'matched_1001_1010.root',
 'n_files': 100,
 'device': 'cpu',
 'WINDOW': 3,
 'USE_TIMING': False,
 'POS_REF': 'seed',
 'VERTEX_Z_MAX': 100.0}

- `sig_flux_prod_vertex_z < VERTEX_Z_MAX` keeps photons produced near the interaction point and drops material-conversion candidates.\n- Events whose entries share one reconstructed cluster are ambiguous (identical cells, different truth energies); drop the whole event.

In [2]:
with uproot.open(INPUT_FILE) as f:
    cuts = f[TREE_NAME].arrays(
        ["sig_flux_prod_vertex_z", "event", "x_cluster", "y_cluster", "sig_flux_eTot"], library="ak")

vz = ak.to_numpy(cuts["sig_flux_prod_vertex_z"]).astype(float)
ev = ak.to_numpy(cuts["event"]).astype(int)
xc = ak.to_numpy(cuts["x_cluster"]).astype(float)
yc = ak.to_numpy(cuts["y_cluster"]).astype(float)
et = ak.to_numpy(cuts["sig_flux_eTot"]).astype(float)
n_total = len(ev)

vertex_ok = vz < VERTEX_Z_MAX

keys = list(zip(ev.tolist(), np.round(xc, 2).tolist(), np.round(yc, 2).tolist()))
counts = collections.Counter(keys)
shared_keys = {k for k, c in counts.items() if c > 1}
bad_events = {k[0] for k in shared_keys}
not_dup = np.array([e not in bad_events for e in ev])

keep = vertex_ok & not_dup
keep_idx = np.flatnonzero(keep)

pd.DataFrame([
    {"step": "total entries", "n": n_total, "frac": 1.0},
    {"step": "pass vertex cut", "n": int(vertex_ok.sum()), "frac": round(float(vertex_ok.mean()), 4)},
    {"step": "in duplicate event", "n": int((~not_dup).sum()), "frac": round(float((~not_dup).mean()), 4)},
    {"step": "kept (both cuts)", "n": int(keep.sum()), "frac": round(float(keep.mean()), 4)},
])

,step,n,frac
0,total entries,1791,1.0000
1,pass vertex cut,350,0.1954
2,in duplicate event,1278,0.7136
3,kept (both cuts),350,0.1954


## Duplicate events being removed (all files, polars)

Survey across all ROOT files of the clusters the duplicate cut drops: one reconstructed cluster matched to several true photons (same cells, different energy target). Built, shown, and exported with polars. The dataset cells below still build from one file; this is the full-sample audit of what the cut removes.

In [3]:
frames = []
for path in root_files:
    with uproot.open(path) as f:
        a = f[TREE_NAME].arrays(
            ["event", "x_cluster", "y_cluster", "sig_flux_eTot", "sig_flux_prod_vertex_z"],
            library="ak")
    ne = len(a["event"])
    frames.append(pl.DataFrame({
        "file": np.full(ne, path.name),
        "entry": np.arange(ne, dtype=np.int64),
        "event": ak.to_numpy(a["event"]).astype(np.int64),
        "cluster_x": np.round(ak.to_numpy(a["x_cluster"]).astype(float), 2),
        "cluster_y": np.round(ak.to_numpy(a["y_cluster"]).astype(float), 2),
        "target_GeV": np.round(ak.to_numpy(a["sig_flux_eTot"]).astype(float), 3),
        "vertex_z": np.round(ak.to_numpy(a["sig_flux_prod_vertex_z"]).astype(float), 1),
    }))
alldf = pl.concat(frames)
{"files": len(root_files), "total_entries": alldf.height}

{'files': 100, 'total_entries': 199538}

In [4]:
dups = (
    alldf.group_by(["file", "event", "cluster_x", "cluster_y"])
         .agg(
             pl.len().alias("n_entries"),
             pl.col("entry").alias("entry_indices"),
             pl.col("target_GeV").alias("targets_GeV"),
             pl.col("target_GeV").n_unique().alias("n_distinct_targets"),
             pl.col("vertex_z").min().alias("vertex_z_min"),
             pl.col("vertex_z").max().alias("vertex_z_max"),
         )
         .filter(pl.col("n_entries") > 1)
         .sort("n_entries", descending=True)
)
dups.head(12)

file,event,cluster_x,cluster_y,n_entries,entry_indices,targets_GeV,n_distinct_targets,vertex_z_min,vertex_z_max
str,i64,f64,f64,u32,list[i64],list[f64],u32,f64,f64
"""matched_1491_1500.root""",967,548.55,-1279.95,23,"[2042, 2043, … 2064]","[1.087, 20.346, … 2.549]",23,1694.1,12271.4
"""matched_1691_1700.root""",584,-304.75,-914.25,20,"[1188, 1189, … 1207]","[8.006, 6.374, … 1.093]",20,339.0,12272.4
"""matched_1691_1700.root""",281,-421.34,63.1,19,"[585, 586, … 603]","[14.595, 1.915, … 2.142]",19,1661.4,12247.3
"""matched_1521_1530.root""",405,63.59,299.74,17,"[895, 896, … 911]","[4.577, 1.605, … 22.164]",16,10359.3,12265.2
"""matched_1181_1190.root""",688,63.59,299.74,17,"[1357, 1358, … 1373]","[1.224, 12.02, … 1.274]",17,8710.9,12242.3
…,…,…,…,…,…,…,…,…,…
"""matched_1831_1840.root""",982,-2620.85,-60.95,16,"[1972, 1973, … 1987]","[1.372, 1.649, … 1.01]",16,499.4,12244.9
"""matched_1721_1730.root""",24,-299.34,-63.09,16,"[56, 57, … 71]","[1.788, 2.068, … 1.549]",16,8577.5,12270.1
"""matched_1301_1310.root""",783,-299.34,-63.09,16,"[1535, 1536, … 1550]","[30.029, 2.048, … 4.533]",16,9467.4,12260.6


In [5]:
per_file = (
    alldf.group_by("file").agg(pl.len().alias("entries"))
         .join(
             dups.group_by("file").agg(
                 pl.col("n_entries").sum().alias("dup_entries"),
                 pl.len().alias("dup_groups"),
             ),
             on="file", how="left",
         )
         .with_columns(pl.col("dup_entries").fill_null(0), pl.col("dup_groups").fill_null(0))
         .with_columns((pl.col("dup_entries") / pl.col("entries")).alias("dup_frac"))
         .sort("dup_frac", descending=True)
)
per_file.head(10)

file,entries,dup_entries,dup_groups,dup_frac
str,u32,u32,u32,f64
"""matched_1591_1600.root""",2061,1523,396,0.738962
"""matched_1691_1700.root""",2049,1505,379,0.734505
"""matched_1881_1890.root""",1760,1292,325,0.734091
"""matched_1491_1500.root""",2108,1515,413,0.718691
"""matched_1831_1840.root""",1995,1428,383,0.715789
"""matched_1331_1340.root""",1955,1384,368,0.707928
"""matched_1341_1350.root""",2010,1400,396,0.696517
"""matched_1071_1080.root""",1935,1344,395,0.694574
"""matched_1991_2000.root""",2176,1506,428,0.692096


In [6]:
reports = repo / "reports"
reports.mkdir(exist_ok=True)

dups.write_parquet(reports / "duplicates_removed_all_files.parquet")
(
    dups.with_columns(
        pl.col("entry_indices").cast(pl.List(pl.Utf8)).list.join(","),
        pl.col("targets_GeV").cast(pl.List(pl.Utf8)).list.join(","),
    ).write_csv(reports / "duplicates_removed_all_files.csv")
)
per_file.write_csv(reports / "duplicates_per_file.csv")

{
    "files": len(root_files),
    "total_entries": alldf.height,
    "duplicate_groups": dups.height,
    "entries_in_duplicates": int(dups["n_entries"].sum()),
    "overall_dup_frac": round(int(dups["n_entries"].sum()) / alldf.height, 4),
    "max_group_size": int(dups["n_entries"].max()),
    "groups_all_targets_distinct": int((dups["n_distinct_targets"] == dups["n_entries"]).sum()),
    "saved": ["reports/duplicates_removed_all_files.parquet",
              "reports/duplicates_removed_all_files.csv",
              "reports/duplicates_per_file.csv"],
}

{'files': 100,
 'total_entries': 199538,
 'duplicate_groups': 39787,
 'entries_in_duplicates': 133159,
 'overall_dup_frac': 0.6673,
 'max_group_size': 23,
 'groups_all_targets_distinct': 39754,
 'saved': ['reports/duplicates_removed_all_files.parquet',
  'reports/duplicates_removed_all_files.csv',
  'reports/duplicates_per_file.csv']}

## Geometry and seed (reused from notebook 01)

In [7]:
_PITCH = np.array([15.0, 30.0, 40.0, 60.0, 120.0])

def derive_geom(c):
    x = c["cell_x"]; y = c["cell_y"]
    ix = c["imodx"]; iy = c["jmody"]
    pts = np.stack([x, y], 1)
    pitch = np.full(len(x), np.nan)
    for key in {(int(a), int(b)) for a, b in zip(ix, iy)}:
        sel = (ix == key[0]) & (iy == key[1])
        p = pts[sel]
        if len(p) >= 2:
            d = np.sqrt(((p[:, None, :] - p[None, :, :]) ** 2).sum(-1))
            d[d == 0] = np.inf
            pitch[sel] = np.median(np.min(d, axis=1))
    fill = np.nanmedian(pitch) if np.isfinite(pitch).any() else 120.0
    pitch[~np.isfinite(pitch)] = fill
    seed = int(np.nanargmax(c["energy"]))
    rel_x = x - x[seed]; rel_y = y - y[seed]
    rel_dr = np.hypot(rel_x, rel_y)
    mod = np.array([int(np.argmin(np.abs(_PITCH - p))) for p in pitch], dtype=float)
    return pitch, mod, rel_x, rel_y, rel_dr, seed

## Cell selection — N x N window around the seed

Keeps the seed cell plus `WINDOW // 2` rings in each axis, measured in seed-pitch units. This caps tokens per cluster at `WINDOW**2` instead of up to ~528, which is the main memory reduction.

In [8]:
def select_window(c, window):
    pitch, mod, rel_x, rel_y, rel_dr, seed = derive_geom(c)
    ps = pitch[seed]
    ix = np.round(rel_x / ps)
    iy = np.round(rel_y / ps)
    half = window // 2
    return (np.abs(ix) <= half) & (np.abs(iy) <= half)

## Token features — timing optional, position reference configurable

In [9]:
CELL_KEYS = ["cell_x", "cell_y", "cell_energies_front", "cell_energies_back", "energy",
             "cell_times_front", "cell_times_back", "imodx", "jmody"]
THRESH_TIME = 1e6
CONT_DIM = 9 if USE_TIMING else 7

def raw_token_features(c, ref_xy=None):
    e = c["energy"]; fr = c["cell_energies_front"]; bk = c["cell_energies_back"]
    pitch, mod, rel_x, rel_y, rel_dr, seed = derive_geom(c)
    if ref_xy is not None:
        rel_x = c["cell_x"] - ref_xy[0]
        rel_y = c["cell_y"] - ref_xy[1]
        rel_dr = np.hypot(rel_x, rel_y)
    cols = [np.log1p(np.clip(e, 0, None)),
            np.log1p(np.clip(fr, 0, None)),
            np.log1p(np.clip(bk, 0, None)),
            rel_x / pitch, rel_y / pitch, rel_dr / pitch,
            np.log(pitch)]
    flag_cols = []
    if USE_TIMING:
        tf = c["cell_times_front"]; tb = c["cell_times_back"]
        vf = np.abs(tf) < THRESH_TIME; vb = np.abs(tb) < THRESH_TIME
        tf = np.where(vf, tf, 0.0); tb = np.where(vb, tb, 0.0)
        cols += [tf, tb]
        flag_cols.append((vf & vb).astype(float)[:, None])
    cont = np.stack(cols, 1)
    onehot = np.zeros((len(e), len(_PITCH)))
    onehot[np.arange(len(e)), mod.astype(int)] = 1.0
    flags = np.concatenate([onehot] + flag_cols, 1)
    return cont, flags

## PyTorch dataset — cuts + window applied

In [10]:
class ShowerWindowDataset(Dataset):
    def __init__(self, path, keep_idx, window=WINDOW, pos_ref=POS_REF):
        with uproot.open(path) as f:
            self.a = f[TREE_NAME].arrays(
                CELL_KEYS + ["x_cluster", "y_cluster", "sig_flux_eTot"], library="ak")
        self.idx = np.asarray(keep_idx)
        self.window = window
        self.pos_ref = pos_ref
        self.mean = None
        self.std = None

    def __len__(self):
        return len(self.idx)

    def _cells(self, i):
        return {k: np.asarray(ak.to_numpy(self.a[k][i])).astype(float) for k in CELL_KEYS}

    def _ref(self, i):
        if self.pos_ref == "cluster":
            return (float(self.a["x_cluster"][i]), float(self.a["y_cluster"][i]))
        return None

    def _windowed(self, i):
        c = self._cells(i)
        sel = select_window(c, self.window)
        return {k: v[sel] for k, v in c.items()}

    def fit_norm(self, n_sample=400):
        chunks = []
        for j in range(min(n_sample, len(self))):
            i = int(self.idx[j])
            chunks.append(raw_token_features(self._windowed(i), self._ref(i))[0])
        allc = np.concatenate(chunks, 0)
        self.mean = allc.mean(0)
        self.std = allc.std(0) + 1e-6
        return self

    def __getitem__(self, j):
        i = int(self.idx[j])
        cw = self._windowed(i)
        cont, flags = raw_token_features(cw, self._ref(i))
        if self.mean is not None:
            cont = (cont - self.mean) / self.std
        feats = np.concatenate([cont, flags], 1)
        y = float(np.log(max(float(self.a["sig_flux_eTot"][i]), 1e-3)))
        return torch.tensor(feats, dtype=torch.float32), torch.tensor([y], dtype=torch.float32)

ds = ShowerWindowDataset(INPUT_FILE, keep_idx).fit_norm()
f0, y0 = ds[0]
{"kept_clusters": len(ds), "token_dim_F": int(f0.shape[1]), "cont_dim": CONT_DIM,
 "cells_in_window": int(f0.shape[0]), "target_logE": round(y0.item(), 3)}

{'kept_clusters': 350,
 'token_dim_F': 12,
 'cont_dim': 7,
 'cells_in_window': 9,
 'target_logE': 3.12}

## Masked batch

In [11]:
def collate(batch):
    feats, ys = zip(*batch)
    Lmax = max(f.shape[0] for f in feats)
    F = feats[0].shape[1]
    B = len(batch)
    X = torch.zeros(B, Lmax, F)
    mask = torch.zeros(B, Lmax, dtype=torch.bool)
    for i, f in enumerate(feats):
        X[i, : f.shape[0]] = f
        mask[i, : f.shape[0]] = True
    return X.to(DEVICE), mask.to(DEVICE), torch.stack(ys).to(DEVICE)

loader = DataLoader(ds, batch_size=16, shuffle=True, collate_fn=collate)
Xb, mb, Yb = next(iter(loader))
{"X": tuple(Xb.shape), "mask": tuple(mb.shape), "Y": tuple(Yb.shape),
 "cells_per_row": mb.sum(1).tolist()}

{'X': (16, 10, 12),
 'mask': (16, 10),
 'Y': (16, 1),
 'cells_per_row': [9, 6, 9, 6, 9, 6, 9, 9, 9, 9, 9, 9, 10, 9, 9, 9]}

## Memory reduction from the window

In [12]:
with uproot.open(INPUT_FILE) as f:
    full = f[TREE_NAME].arrays(["energy"], library="ak")
full_cells = ak.to_numpy(ak.num(full["energy"], axis=1))
win_cells = []
for j in range(min(500, len(ds))):
    i = int(ds.idx[j])
    win_cells.append(int(select_window(ds._cells(i), WINDOW).sum()))
win_cells = np.array(win_cells)
pd.DataFrame([
    {"set": "full cluster", "max_cells": int(full_cells.max()), "median_cells": int(np.median(full_cells))},
    {"set": f"{WINDOW}x{WINDOW} window", "max_cells": int(win_cells.max()), "median_cells": int(np.median(win_cells))},
])

,set,max_cells,median_cells
0,full cluster,528,132
1,3x3 window,24,9


## Tiny transformer smoke step (CPU)

In [13]:
class TinyShowerTransformer(nn.Module):
    def __init__(self, in_dim, d_model=32, nhead=4, layers=2):
        super().__init__()
        self.embed = nn.Linear(in_dim, d_model)
        layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=64, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x, mask):
        h = self.enc(self.embed(x), src_key_padding_mask=~mask)
        w = mask.unsqueeze(-1).float()
        pooled = (h * w).sum(1) / w.sum(1).clamp(min=1)
        return self.head(pooled)

torch.manual_seed(0)
model = TinyShowerTransformer(Xb.shape[2]).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
pred = model(Xb, mb)
loss0 = nn.functional.mse_loss(pred, Yb)
opt.zero_grad(); loss0.backward(); opt.step()
loss1 = nn.functional.mse_loss(model(Xb, mb), Yb)
{"output": tuple(pred.shape), "loss": round(loss0.item(), 4), "after_1_step": round(loss1.item(), 4)}

{'output': (16, 1), 'loss': 10.0747, 'after_1_step': 9.2276}

## Summary

In [14]:
{
    "cuts": {"vertex_z<": VERTEX_Z_MAX, "drop_duplicate_events": True},
    "kept_clusters": len(ds),
    "cell_window": f"{WINDOW}x{WINDOW} around seed",
    "token_dim_F": int(Xb.shape[2]),
    "timing_features": USE_TIMING,
    "position_reference": POS_REF,
    "target": "log(sig_flux_eTot) single photon",
    "pooling": "masked mean",
    "device": str(DEVICE),
    "next": "swap WINDOW to 5, POS_REF to cluster, or USE_TIMING on, then compare",
}

{'cuts': {'vertex_z<': 100.0, 'drop_duplicate_events': True},
 'kept_clusters': 350,
 'cell_window': '3x3 around seed',
 'token_dim_F': 12,
 'timing_features': False,
 'position_reference': 'seed',
 'target': 'log(sig_flux_eTot) single photon',
 'pooling': 'masked mean',
 'device': 'cpu',
 'next': 'swap WINDOW to 5, POS_REF to cluster, or USE_TIMING on, then compare'}